# HSD Ensemble: PhoBERT + ViHateT5 + Qwen (Hard Voting)
**Models:** `KalvinPhan/phobert-vihsd-finetuned` · `tarudesu/ViHateT5-base-HSD`  
**Data:** `/content/tiktok.xlsx` · `/content/qwen(zero-shot).json`

In [ ]:
# ============================================================
# CELL 1 – Install dependencies
# ============================================================
!pip install -q transformers sentencepiece openpyxl accelerate

In [ ]:
# ============================================================
# CELL 2 – Imports & device
# ============================================================
import json, re
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM
)

from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [ ]:
# ============================================================
# CELL 3 – Load data
# ============================================================
df_tiktok = pd.read_excel('/content/tiktok_add.xlsx')
print('tiktok.xlsx shape:', df_tiktok.shape)
print('Columns:', df_tiktok.columns.tolist())
print(df_tiktok.head(3))


tiktok.xlsx shape: (8909, 5)
Columns: ['text', 'topic', 'keyword', 'post_url', 'predicted_label']
                                                text     topic  keyword  \
0  m3p đáng thương thật. vừa đi 1 cái psg nó lụm ...  the_thao  Bóng đá   
1  nếu lỡ năm nay bồ vô địch world cup chắc vitin...  the_thao  Bóng đá   
2       ko có quả pen chưa thì chưa chắc psg vô địch  the_thao  Bóng đá   

                                            post_url  predicted_label  
0  https://www.tiktok.com/@hadezkjc/video/7645775...                0  
1  https://www.tiktok.com/@hadezkjc/video/7645775...                0  
2  https://www.tiktok.com/@hadezkjc/video/7645775...                0  


In [ ]:
# ============================================================
# CELL 5 – Load PhoBERT model
# ============================================================
PHOBERT_NAME = 'KalvinPhan/phobert-vihsd-finetuned'
print('Loading PhoBERT tokenizer & model...')
phobert_tokenizer = AutoTokenizer.from_pretrained(PHOBERT_NAME)
phobert_model     = AutoModelForSequenceClassification.from_pretrained(PHOBERT_NAME)
phobert_model     = phobert_model.to(device)
phobert_model.eval()
print('PhoBERT id2label:', phobert_model.config.id2label)

Loading PhoBERT tokenizer & model...


config.json:   0%|          | 0.00/969 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PhoBERT id2label: {0: 'Clean', 1: 'Offensive', 2: 'Hate'}


In [ ]:

# ============================================================
# CELL 6 – Load ViHateT5 model
# ============================================================
VIHATE_NAME = 'tarudesu/ViHateT5-base-HSD'
print('Loading ViHateT5 tokenizer & model...')
vihate_tokenizer = AutoTokenizer.from_pretrained(VIHATE_NAME)
vihate_model     = AutoModelForSeq2SeqLM.from_pretrained(VIHATE_NAME)
vihate_model     = vihate_model.to(device)
vihate_model.eval()
print('ViHateT5 loaded.')

Loading ViHateT5 tokenizer & model...


config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.05M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

ViHateT5 loaded.


In [ ]:
# ============================================================
# CELL 7 – Inference functions
# ============================================================

# ---------- PhoBERT ----------
def predict_phobert(texts, batch_size=16):
    """
    Returns list of int labels (0=CLEAN, 1=OFFENSIVE, 2=HATE)
    using the model's own id2label mapping.
    """
    id2label = phobert_model.config.id2label
    # Build a consistent text→int mapping regardless of label string casing
    label2int = {}
    for idx, lbl in id2label.items():
        lbl_up = lbl.upper()
        if 'HATE' in lbl_up and 'OFFENSIVE' not in lbl_up:
            label2int[idx] = 2
        elif 'OFFENSIVE' in lbl_up or 'OFF' in lbl_up:
            label2int[idx] = 1
        else:
            label2int[idx] = 0

    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = phobert_tokenizer(
            batch, padding=True, truncation=True,
            max_length=256, return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            logits = phobert_model(**enc).logits
        preds = logits.argmax(dim=-1).cpu().tolist()
        all_preds.extend([label2int.get(p, p) for p in preds])
    return all_preds


# ---------- ViHateT5 ----------
VIHATE_PREFIX = 'hate-speech-detection'

def _vihate_output_to_label(output_text):
    """Map ViHateT5 decoded string → 0/1/2."""
    t = output_text.strip().upper()
    if 'HATE' in t and 'OFFENSIVE' not in t:
        return 2
    elif 'OFFENSIVE' in t or 'OFF' in t:
        return 1
    else:
        return 0

def predict_vihate(texts, batch_size=8):
    """Returns list of int labels (0/1/2)."""
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        prefixed = [VIHATE_PREFIX + ': ' + t for t in batch]
        enc = vihate_tokenizer(
            prefixed, padding=True, truncation=True,
            max_length=256, return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            out_ids = vihate_model.generate(
                enc['input_ids'],
                attention_mask=enc['attention_mask'],
                max_length=32
            )
        decoded = vihate_tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        all_preds.extend([_vihate_output_to_label(d) for d in decoded])
    return all_preds

In [ ]:
# ============================================================
# CELL 8 – Run inference on tiktok.xlsx
# ============================================================
texts = df_tiktok['text'].astype(str).tolist()

# Lấy preds của Qwen từ file kết quả bước trước (giả sử tên cột là 'predicted_label')
if 'predicted_label' in df_tiktok.columns:
    qwen_preds = df_tiktok['predicted_label'].astype(int).tolist()
else:
    print("⚠️ CẢNH BÁO: Không tìm thấy cột 'predicted_label' của Qwen trong file csv!")
    qwen_preds = [-1] * len(texts) # Tránh lỗi code nếu thiếu cột

print(f'Total samples: {len(texts)}')

print('\n[1/2] Running PhoBERT...')
phobert_preds = predict_phobert(texts)
print('PhoBERT done. Label distribution:', Counter(phobert_preds))

print('\n[2/2] Running ViHateT5...')
vihate_preds = predict_vihate(texts)
print('ViHateT5 done. Label distribution:', Counter(vihate_preds))

Total samples: 8909

[1/2] Running PhoBERT...
PhoBERT done. Label distribution: Counter({0: 8664, 1: 197, 2: 48})

[2/2] Running ViHateT5...
ViHateT5 done. Label distribution: Counter({0: 8798, 1: 78, 2: 33})


In [ ]:
from google.colab import files
from collections import Counter

# ============================================================
# CELL 10 – Hard Voting & Save File
# ============================================================
def hard_vote(votes):
    """Return majority label; tie → first model (PhoBERT) wins."""
    count = Counter(votes)
    max_count = max(count.values())
    candidates = [k for k, v in count.items() if v == max_count]
    # Prefer the order: phobert, vihate, qwen on tie
    for v in votes:
        if v in candidates:
            return v

# Chạy Hard Voting
ensemble_preds = [
    hard_vote([pb, vh, qw])
    for pb, vh, qw in zip(phobert_preds, vihate_preds, qwen_preds)
]
print('Ensemble distribution:', Counter(ensemble_preds))

# --- BƯỚC LƯU FILE ---
# Tạo cột kết quả cuối cùng
df_tiktok['hard_voting_label'] = ensemble_preds

# (Tùy chọn) Xóa cột predicted_label của Qwen nếu bạn chỉ muốn giữ lại nhãn cuối cùng
# df_tiktok = df_tiktok.drop(columns=['predicted_label'])

# Đường dẫn lưu trên Colab (thư mục /content/)
output_path = '/content/threads_final_hardvoting.xlsx'

# Xuất excel
df_tiktok.to_excel(output_path, index=False)

print(f"\nĐã lưu thành công file kết quả tổng hợp tại: {output_path}")

# Tự động tải file về máy cá nhân
print("Đang tiến hành tải file về máy...")
files.download(output_path)

Ensemble distribution: Counter({0: 8756, 1: 126, 2: 27})

Đã lưu thành công file kết quả tổng hợp tại: /content/threads_final_hardvoting.xlsx
Đang tiến hành tải file về máy...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>